# Week 4: Backpropagation, SGD, and Regularization

**Lecture 7:** Backpropagation implementation  
**Lecture 8:** Stochastic gradient descent and regularization

# Lecture 7: Implementing Backpropagation

This implementation follows the equations from Week 3. The forward pass stores every activation. The backward pass computes the output delta, propagates deltas backward, and constructs explicit gradients for each $W^{(\ell)}$ and $b^{(\ell)}$.

Lecture 7 deliberately uses **full-batch gradient descent**: every update is computed from the complete training set. Lecture 8 will upgrade the class to use mini-batch SGD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
class MultilayerPerceptron:
    def __init__(self, layer_sizes, alpha=0.5, seed=1):
        self.alpha = alpha
        self.rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2 / (input_size + output_size))
            self.weights.append(
                self.rng.normal(0, scale, size=(input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def sigmoid_gradient(a):
        return a * (1 - a)

    @staticmethod
    def softmax(z):
        shifted_z = z - z.max(axis=1, keepdims=True)
        exp_z = np.exp(shifted_z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def forward(self, X):
        activations = [np.atleast_2d(X)]

        for layer, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = activations[-1] @ W + b
            if layer == len(self.weights) - 1:
                a = self.softmax(z)
            else:
                a = self.sigmoid(z)
            activations.append(a)

        return activations

    def cross_entropy(self, X, y):
        probabilities = np.clip(self.forward(X)[-1], 1e-12, 1.0)
        return -np.mean(np.sum(y * np.log(probabilities), axis=1))

    def gradients(self, X, y):
        X = np.atleast_2d(X)
        y = np.atleast_2d(y)
        n = X.shape[0]
        activations = self.forward(X)

        weight_gradients = [None] * len(self.weights)
        bias_gradients = [None] * len(self.biases)

        # Softmax plus cross-entropy gives this explicit output delta.
        delta = activations[-1] - y

        for layer in range(len(self.weights) - 1, -1, -1):
            weight_gradients[layer] = activations[layer].T @ delta / n
            bias_gradients[layer] = delta.mean(axis=0)

            if layer > 0:
                delta = (
                    delta @ self.weights[layer].T
                    * self.sigmoid_gradient(activations[layer])
                )

        return weight_gradients, bias_gradients

    def step(self, X, y):
        weight_gradients, bias_gradients = self.gradients(X, y)

        for layer in range(len(self.weights)):
            self.weights[layer] -= self.alpha * weight_gradients[layer]
            self.biases[layer] -= self.alpha * bias_gradients[layer]

    def fit(self, X, y, epochs=100, update=10):
        history = []

        for epoch in range(epochs):
            # One update from the entire training set.
            self.step(X, y)

            if (epoch + 1) % update == 0 or epoch == 0:
                history.append((epoch + 1, self.cross_entropy(X, y)))

        return history

    def predict_proba(self, X):
        return self.forward(X)[-1]

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

### Tiny MNIST Example

Scikit-learn's digits dataset is a small, built-in analogue of MNIST. It contains 1,797 grayscale images at only $8\times8$ pixels, so we can inspect the full-batch implementation quickly before moving to $28\times28$ MNIST.

In [ ]:
digits = load_digits()
X_tiny = digits.data / 16.0
y_tiny = np.eye(10)[digits.target]

X_tiny_train, X_tiny_test, y_tiny_train, y_tiny_test = train_test_split(
    X_tiny,
    y_tiny,
    test_size=0.25,
    random_state=1,
    stratify=digits.target,
)

tiny_model = MultilayerPerceptron([64, 32, 10], alpha=1.0, seed=1)
tiny_history = tiny_model.fit(
    X_tiny_train, y_tiny_train, epochs=1_000, update=100
)

print("training history:", tiny_history)
print(
    classification_report(
        y_tiny_test.argmax(axis=1), tiny_model.predict(X_tiny_test)
    )
)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
predictions = tiny_model.predict(X_tiny_test[:10])

for image, prediction, target, ax in zip(
    X_tiny_test[:10],
    predictions,
    y_tiny_test[:10].argmax(axis=1),
    axes.ravel(),
):
    ax.imshow(image.reshape(8, 8), cmap="gray")
    ax.set_title(f"pred={prediction}, true={target}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### MNIST with 1,000 Training Images

MNIST contains $28\times28$ images, giving 784 input features. To expose the cost and limitations of full-batch learning, Lecture 7 trains on only the first 1,000 training images. The test set remains separate.

In [ ]:
with np.load('../../datasets/mnist.npz') as mnist_data:
    X_mnist_train_full = mnist_data['x_train']
    y_mnist_train_full = mnist_data['y_train']
    X_mnist_test = mnist_data['x_test']
    y_mnist_test = mnist_data['y_test']

X_mnist_train = (
    X_mnist_train_full[:1_000].reshape(1_000, 28 * 28).astype("float32") / 255
)
y_mnist_train = np.eye(10)[y_mnist_train_full[:1_000]]

# A 2,000-image test subset keeps this instructional run quick.
X_mnist_test_small = (
    X_mnist_test[:2_000].reshape(2_000, 28 * 28).astype("float32") / 255
)
y_mnist_test_small = y_mnist_test[:2_000]

mnist_1k_model = MultilayerPerceptron([784, 32, 10], alpha=1.0, seed=1)
mnist_1k_history = mnist_1k_model.fit(
    X_mnist_train, y_mnist_train, epochs=200, update=20
)

print("training history:", mnist_1k_history)
print(
    classification_report(
        y_mnist_test_small,
        mnist_1k_model.predict(X_mnist_test_small),
    )
)

# Lecture 8: Stochastic Gradient Descent and Regularization

Full-batch gradient descent computes one update from all $n$ examples. **Stochastic gradient descent (SGD)** estimates the gradient from a shuffled mini-batch $B$:

$$\nabla L_B(w)=\frac{1}{|B|}\sum_{i\in B}\nabla L_i(w).$$

The upgraded class below reuses the same explicit backpropagation equations but makes many smaller updates per epoch.

It also supports L2 regularization (weight decay):

$$L_{\mathrm{regularized}}(w)=L_{\mathrm{data}}(w)+\frac{\lambda}{2}\sum_{\ell}\|W^{(\ell)}\|_F^2.$$

The explicit weight gradient therefore gains $\lambda W^{(\ell)}$. Biases are not penalized.

In [ ]:
class StochasticMultilayerPerceptron(MultilayerPerceptron):
    def __init__(
        self,
        layer_sizes,
        alpha=0.1,
        batch_size=64,
        l2=0.0,
        seed=1,
    ):
        super().__init__(layer_sizes, alpha=alpha, seed=seed)
        self.batch_size = batch_size
        self.l2 = l2

    def cross_entropy(self, X, y):
        data_loss = super().cross_entropy(X, y)
        penalty = 0.5 * self.l2 * sum(
            np.sum(W**2) for W in self.weights
        )
        return data_loss + penalty

    def gradients(self, X, y):
        weight_gradients, bias_gradients = super().gradients(X, y)

        for layer in range(len(weight_gradients)):
            weight_gradients[layer] += self.l2 * self.weights[layer]

        return weight_gradients, bias_gradients

    def fit(self, X, y, epochs=10, update=1):
        X = np.asarray(X)
        y = np.asarray(y)
        history = []

        for epoch in range(epochs):
            indices = self.rng.permutation(len(X))

            for start in range(0, len(X), self.batch_size):
                batch = indices[start : start + self.batch_size]
                self.step(X[batch], y[batch])

            if (epoch + 1) % update == 0 or epoch == 0:
                history.append((epoch + 1, self.cross_entropy(X, y)))

        return history

### Full MNIST with Mini-batch SGD

The upgraded class now trains on all 60,000 MNIST training images. Each epoch uses every image once, but updates occur after each shuffled mini-batch rather than after the entire dataset.

In [ ]:
X_mnist_train_full = (
    X_mnist_train_full.reshape(60_000, 28 * 28).astype("float32") / 255
)
X_mnist_test_full = (
    X_mnist_test.reshape(10_000, 28 * 28).astype("float32") / 255
)
y_mnist_train_full_one_hot = np.eye(10)[y_mnist_train_full]

sgd_model = StochasticMultilayerPerceptron(
    [784, 32, 16, 10],
    alpha=0.5,
    batch_size=64,
    l2=1e-5,
    seed=1,
)
sgd_history = sgd_model.fit(
    X_mnist_train_full,
    y_mnist_train_full_one_hot,
    epochs=10,
    update=1,
)

print("training history:", sgd_history)
print(classification_report(y_mnist_test, sgd_model.predict(X_mnist_test_full)))